# NB11a — external cohort ds004584: download, fixed preamble, features, PSWE, frozen folds (CPU)

Same code path as NB01–NB03 with the `configs/ds004584.yaml` overlay (Pz reference restored, 60 Hz notch, eyes-open task, PD/Control labels, MoCA as the analysis-only cognitive score). The fold file is committed to the repository before any training run on this cohort (NB11b).

In [ ]:
REPO = 'https://github.com/Devguru-codes/Major_project_IIITN.git'
BRANCH = 'feat/implementation'
WORK = '/kaggle/working'
SRC = '/tmp/eegrep-src'   # code checkout stays out of the saved output

import subprocess, sys, os, json, platform, shutil

def sh(cmd, log=None):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-20000:], r.stderr[-5000:], sep='\n', flush=True)
    if log:
        open(log, 'w').write(r.stdout + '\n' + r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f'command failed ({r.returncode}): {cmd}')
    return r.stdout

shutil.rmtree(SRC, ignore_errors=True)
sh(f'git clone --depth 1 -b {BRANCH} {REPO} {SRC}')
SHA = sh(f'git -C {SRC} rev-parse --short HEAD').strip()
sh(f'pip install -q -e "{SRC}[preprocess,dev,stats]"', log=f'{WORK}/pip_install.log')
PY = f'cd {SRC} && {sys.executable} -m eegrep'
print('git sha', SHA)

In [ ]:
sh(f'cd {SRC} && {sys.executable} -m pytest -q -p no:cacheprovider', log=f'{WORK}/pytest.txt')

In [ ]:
EXT = f'{PY} --config configs/ds004584.yaml'
sh(f'{EXT} download --dest /tmp/ds004584', log=f'{WORK}/download.log')
shutil.copy('/tmp/ds004584/download_manifest.json', f'{WORK}/download_manifest.json')
shutil.copy('/tmp/ds004584/participants.tsv', f'{WORK}/participants.tsv')

In [ ]:
EXT = f'{PY} --config configs/ds004584.yaml'
sh(f'{EXT} preprocess --bids /tmp/ds004584 --out {WORK}/preproc --n-jobs 4', log=f'{WORK}/preprocess.log')

In [ ]:
EXT = f'{PY} --config configs/ds004584.yaml'
sh(f'{EXT} cache --preproc {WORK}/preproc --participants {WORK}/participants.tsv --out {WORK}/cache', log=f'{WORK}/cache.log')
sh(f'{EXT} qc --cache {WORK}/cache --logs {WORK}/preproc/preprocess_log.jsonl --out {WORK}/qc_report.json --no-fail')
print(json.load(open(f'{WORK}/qc_report.json'))['gate_failures'])

In [ ]:
EXT = f'{PY} --config configs/ds004584.yaml'
sh(f'{EXT} folds --participants {WORK}/participants.tsv --out {WORK}/folds_ds004584.json')
sh(f'{EXT} demographics --participants {WORK}/participants.tsv --folds {WORK}/folds_ds004584.json --out {WORK}/demographics.csv', log=f'{WORK}/demographics.txt')

In [ ]:
EXT = f'{PY} --config configs/ds004584.yaml'
sh(f'{EXT} pswe-stats --cache {WORK}/cache --participants {WORK}/participants.tsv --out {WORK}/rq6', log=f'{WORK}/pswe_stats.txt')
sh(f'{EXT} pswe-relative --preproc {WORK}/preproc --cache {WORK}/cache --participants {WORK}/participants.tsv --out {WORK}/rq6_relative', log=f'{WORK}/pswe_relative.txt')

In [ ]:
import torch
sh(f'{sys.executable} -m pip freeze', log=f'{WORK}/pip_freeze.txt')
env = {'git_sha': SHA, 'python': platform.python_version(), 'torch': torch.__version__,
       'cuda': torch.cuda.is_available(), 'cpu_count': os.cpu_count(),
       'kaggle_image': os.environ.get('KAGGLE_DOCKER_IMAGE', 'unknown')}
json.dump(env, open(f'{WORK}/environment.json', 'w'), indent=1)
print(env)